# Fine-Tuning

In [85]:
data = [
  "He composes songs and practices piano daily.",
  "She reads books and explores the nearby caves."
]

In [86]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMBED_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [87]:
import os
import re
import torch
import json
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader


In [88]:
save_dir = "models"

In [89]:
_ = torch.manual_seed(123)

In [109]:
def clean_text(s):
    s = s.lower()
    s = re.sub(r'([.,!?])', r' \1 ', s)   # separate ALL punctuation
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

In [107]:
file_name = "vocab.json"
with open(os.path.join(save_dir,file_name)) as f:
   vocab = json.load(f)

In [108]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: He
1: She
2: and
3: art
4: books
5: builds
6: caves.
7: climbs
8: collaborates
9: complex
10: composes
11: creative
12: curates
13: daily.
14: designs
15: digital
16: documents
17: every
18: everyday
19: exhibitions
20: experiments
21: explores
22: fairs.
23: filmmakers.
24: for
25: friends.
26: harmonies
27: her
28: in
29: jewelry
30: local
31: maps
32: marathons.
33: models.
34: mountains
35: music
36: music.
37: navigation
38: nearby
39: newspaper
40: novel
41: novels.
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects.
48: puzzles.
49: reads
50: regularly.
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars.
58: studies
59: teaches
60: the
61: trains
62: trips.
63: tunes
64: using
65: weekend.
66: wildlife
67: with
68: wooden
69: writes


In [ ]:

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}

In [94]:
def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[i] for i in token_ids]
    return ' '.join([id_to_word[id] for id in words])

In [95]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len
       
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len+1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

train_dataset = LLMDataset(
    text_data, 
    word_to_id, 
    CONTEXT_LEN
    )

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
    )
print(train_dataloader)
print(len(train_dataloader))



1


In [96]:
class Attention(nn.Module):
    def __init__(self,d_in,d_out, context_length):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out,bias=False)
        self.W_key = nn.Linear(d_in, d_out,bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length),diagonal=1))
    
    
    def forward(self, x, return_weights=False):
        _, num_tokens, _ = x.shape

        queries = self.query(x)  # (batch_size, seq_len, embed_dim)
        keys = self.key(x)    # (batch_size, seq_len, embed_dim)
        value = self.value(x)  # (batch_size, seq_len, embed_dim)

        attn_scores = queries @ keys.transpose(1,2)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
       
        attn_scores = attn_scores.masked_fill(~mask_bool, -torch.inf)
       
        attn_weights = torch.softmax(attn_scores/(self.d_out **0.5), dim=-1)

        context_vec = attn_weights @ value

        if return_weights:
            return context_vec, attn_weights
        return context_vec


In [97]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = Attention(
            d_in= EMBED_DIM,
            d_out= EMBED_DIM, 
            context_length= CONTEXT_LEN)
        
    def forward(self,x):
        shortcut = x

        x = self.att(x)
        x = x + shortcut
        return x

In [98]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMBED_DIM)
      
        self.trf_block1 = TransformerBlock()
        self.trf_block2 = TransformerBlock()
        self.trf_block3 = TransformerBlock()
        self.trf_block4 = TransformerBlock()
        self.trf_block5 = TransformerBlock()
        self.trf_block6 = TransformerBlock()
        self.output_layer = nn.Linear(EMBED_DIM, VOCAB_SIZE)

    
    def forward(self,in_idx):
        _, seq_len = in_idx.shape

        token_emb = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = token_emb + pos_embeds

        x = self.trf_block1(x)
        x = self.trf_block2(x)
        x = self.trf_block3(x)
        x = self.trf_block4(x)  
        x = self.trf_block5(x)
        x = self.trf_block6(x)

        logits = self.output_layer(x)
        
        return logits

In [99]:
model = GPTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [100]:
#file_name = "parameters.bin"
# model.load_state_dict(torch.load(os.path.join(save_dir,file_name)))

file_name = "parameters.bin"
path = os.path.join(save_dir, file_name)

state_dict = torch.load(path)

fixed_state_dict = {}

for k, v in state_dict.items():
    new_key = k

    # Reverse mapping (IMPORTANT)
    new_key = new_key.replace("token_embedding", "tok_emb")
    new_key = new_key.replace("pos_embedding", "pos_emb")

    new_key = new_key.replace("trm_block", "trf_block")
    new_key = new_key.replace(".attention.", ".att.")

    fixed_state_dict[new_key] = v

model.load_state_dict(fixed_state_dict)

RuntimeError: Error(s) in loading state_dict for GPTModel:
	Missing key(s) in state_dict: "output_layer.bias". 

In [101]:
def train(dataloader, model, criterion, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        logits = model(X)
        loss = criterion(logits.flatten(0, 1), y.flatten())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print("batch : {batch +1} loss: {loss: >7f}")
  

In [102]:
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train(train_dataloader, model, criterion, optimizer)
    print("-------------------------------")
print("Training done!")    

Epoch 1
-------------------------------


KeyError: 'daily'